In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
reform_dir = os.path.join(CUR_DIR, 'TCJA_Ext_Plus_Prod_Gain_1.02', "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54
1,IIT: Pct Change due to behavior,-1.03,-0.79,-0.36,-0.05,0.29,0.66,1.08,1.58,2.20,2.99,0.65,3.73
2,IIT: Pct Change due to macro,2.70,5.27,7.95,10.74,13.65,16.69,19.85,23.14,26.55,30.08,15.75,29.05
3,IIT: Overall Pct Change in taxes,-7.04,-4.48,-1.63,1.23,4.25,7.43,10.80,14.40,18.28,22.52,6.56,22.43
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,8.59,17.81,27.83,38.17,49.06,60.52,72.57,85.28,98.74,113.11,57.16,112.87
6,CIT: Pct Change due to macro,-6.36,-10.72,-14.79,-18.45,-21.83,-24.93,-27.79,-30.41,-32.82,-35.05,-24.22,-37.19
7,CIT: Overall Pct Change in taxes,1.68,5.18,8.93,12.68,16.53,20.50,24.62,28.94,33.51,38.42,19.10,33.70
8,All: Pct Change due to tax rates,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.05
9,All: Pct Change due to behavior,-0.45,0.34,1.35,2.27,3.25,4.30,5.43,6.67,8.08,9.70,4.09,10.48


In [4]:
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])


In [5]:
df_levels = df.loc[8:, df.columns[:-2]]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
8,All: Pct Change due to tax rates,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06
9,All: Pct Change due to behavior,-0.45,0.34,1.35,2.27,3.25,4.30,5.43,6.67,8.08,9.70
10,All: Pct Change due to macro,2.10,4.13,6.21,8.35,10.54,12.80,15.11,17.48,19.90,22.37
11,All: Overall Pct Change in taxes,-6.56,-3.94,-1.04,1.87,4.93,8.16,11.58,15.21,19.13,23.41


In [6]:
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
8,All: Pct Change due to tax rates,-0.41,-0.43,-0.46,-0.48,-0.49,-0.51,-0.54,-0.56,-0.58,-0.60
9,All: Pct Change due to behavior,-0.02,0.02,0.08,0.14,0.20,0.27,0.36,0.46,0.58,0.72
10,All: Pct Change due to macro,0.11,0.22,0.36,0.50,0.65,0.81,1.01,1.21,1.43,1.67
11,All: Overall Pct Change in taxes,-0.33,-0.21,-0.06,0.11,0.30,0.52,0.77,1.05,1.37,1.75


In [7]:
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,All: Pct Change due to tax rates,-0.41,-0.43,-0.46,-0.48,-0.49,-0.51,-0.54,-0.56,-0.58,-0.60,-5.06
9,All: Pct Change due to behavior,-0.02,0.02,0.08,0.14,0.20,0.27,0.36,0.46,0.58,0.72,2.81
10,All: Pct Change due to macro,0.11,0.22,0.36,0.50,0.65,0.81,1.01,1.21,1.43,1.67,7.95
11,All: Overall Pct Change in taxes,-0.33,-0.21,-0.06,0.11,0.30,0.52,0.77,1.05,1.37,1.75,5.27


In [8]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.41,-0.43,-0.46,-0.48,-0.49,-0.51,-0.54,-0.56,-0.58,-0.60,-5.06
9,Rev Change Due to Behavior,-0.02,0.02,0.08,0.14,0.20,0.27,0.36,0.46,0.58,0.72,2.81
10,Rev Change Due to Macro,0.11,0.22,0.36,0.50,0.65,0.81,1.01,1.21,1.43,1.67,7.95
11,Total Revenue Change,-0.33,-0.21,-0.06,0.11,0.30,0.52,0.77,1.05,1.37,1.75,5.27


In [9]:
# Get level changes just for IIT + Payroll
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([4.287, 4.655, 5.007, 5.184, 5.365, 5.573, 5.783, 5.994, 6.227, 6.476])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.37,-0.40,-0.43,-0.44,-0.46,-0.48,-0.49,-0.51,-0.53,-0.55,-4.66
1,Rev Change Due to Behavior,-0.04,-0.04,-0.02,-0.00,0.02,0.04,0.06,0.09,0.14,0.19,0.44
2,Rev Change Due to Macro,0.12,0.25,0.40,0.56,0.73,0.93,1.15,1.39,1.65,1.95,9.11
3,Total Revenue Change,-0.30,-0.21,-0.08,0.06,0.23,0.41,0.62,0.86,1.14,1.46,4.20


In [10]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)

In [11]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.37,-0.42,-0.43,-0.45,-0.47,-0.49,-0.51,-0.53,-0.55,-0.58,-4.81
1,Rev Change Due to Behavior,-0.04,-0.04,-0.02,-0.00,0.02,0.04,0.06,0.10,0.14,0.20,0.46
2,Rev Change Due to Macro,0.12,0.26,0.41,0.57,0.75,0.96,1.19,1.44,1.72,2.03,9.44
3,Total Revenue Change,-0.31,-0.22,-0.08,0.07,0.23,0.43,0.65,0.90,1.19,1.52,4.37


In [12]:
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values

In [13]:
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
0,IIT: Pct Change due to tax rates,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54
1,IIT: Pct Change due to behavior,-1.03,-0.79,-0.36,-0.05,0.29,0.66,1.08,1.58,2.20,2.99
2,IIT: Pct Change due to macro,2.70,5.27,7.95,10.74,13.65,16.69,19.85,23.14,26.55,30.08
3,IIT: Overall Pct Change in taxes,-7.04,-4.48,-1.63,1.23,4.25,7.43,10.80,14.40,18.28,22.52


In [14]:
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100

In [15]:
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034
0,IIT: Pct Change due to tax rates,-0.37,-0.42,-0.43,-0.45,-0.47,-0.49,-0.51,-0.53,-0.55,-0.58
1,IIT: Pct Change due to behavior,-0.04,-0.04,-0.02,-0.00,0.02,0.04,0.06,0.10,0.14,0.20
2,IIT: Pct Change due to macro,0.12,0.26,0.41,0.57,0.75,0.96,1.19,1.44,1.72,2.03
3,IIT: Overall Pct Change in taxes,-0.31,-0.22,-0.08,0.07,0.23,0.43,0.65,0.90,1.19,1.52


In [16]:
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,IIT: Pct Change due to tax rates,-0.37,-0.42,-0.43,-0.45,-0.47,-0.49,-0.51,-0.53,-0.55,-0.58,-4.81
1,IIT: Pct Change due to behavior,-0.04,-0.04,-0.02,-0.00,0.02,0.04,0.06,0.10,0.14,0.20,0.46
2,IIT: Pct Change due to macro,0.12,0.26,0.41,0.57,0.75,0.96,1.19,1.44,1.72,2.03,9.44
3,IIT: Overall Pct Change in taxes,-0.31,-0.22,-0.08,0.07,0.23,0.43,0.65,0.90,1.19,1.52,4.37


In [17]:
# jason's get-around (with weifeng's correction in line 6)

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_reform = result_df_static.loc["Reform", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * (tc_reform + df_levels.loc[1, df_levels.columns[1:]])
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.32,-0.33,-0.34,-0.35,-0.36,-0.37,-0.38,-0.39,-0.41,-3.27
1,Rev Change Due to Behavior,-0.04,-0.04,-0.02,-0.00,0.01,0.04,0.06,0.09,0.13,0.19,0.43
2,Rev Change Due to Macro,0.12,0.24,0.38,0.53,0.71,0.91,1.13,1.37,1.65,1.96,8.99
3,Total Revenue Change,0.07,-0.12,0.03,0.19,0.37,0.58,0.81,1.08,1.39,1.75,6.14


In [18]:
df_levels.to_csv('og_usa_result_w_tcja_prod_1.02.csv')